In [ ]:
# Set ENV 
!pip uninstall -y datasets ragas pyarrow
!pip install --no-cache-dir datasets ragas pyarrow requests pandas
!pip install -U langchain-ollama

# ollama pull nomic-embed-text
!ollama pull nomic-embed-text

# 2. Make sure the main LLM model is also downloaded
!ollama pull llama3
!/usr/local/bin/ollama pull nomic-embed-text
!/usr/local/bin/ollama pull llama3

Found existing installation: datasets 4.8.5
Uninstalling datasets-4.8.5:
  Successfully uninstalled datasets-4.8.5
Found existing installation: ragas 0.4.3
Uninstalling ragas-0.4.3:
  Successfully uninstalled ragas-0.4.3
Found existing installation: pyarrow 24.0.0
Uninstalling pyarrow-24.0.0:
  Successfully uninstalled pyarrow-24.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 21.0 MB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.37.1 requires rich<14,>=10.14.0, but you have rich 14.3.4 which is incompatible.
zsh:1: command not found: ollama
zsh:1: command not found: ollama
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ 

In [5]:
# 1. Install necessary libraries quietly
!pip install ragas langchain-ollama openai datasets pandas -q

import subprocess
import time
import os

print("📥 1. Installing Ollama on the Colab Linux backend...")
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)

print("🚀 2. Launching Ollama Server in the background...")
# We use subprocess.Popen so the server runs asynchronously without blocking the notebook cell
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)  # Wait for the local server port to open

print("🦙 3. Pulling llama3 model (Loading directly into VRAM)...")
subprocess.run(["ollama", "pull", "llama3"])

print("🧬 4. Pulling nomic-embed-text embedding model...")
subprocess.run(["ollama", "pull", "nomic-embed-text"])

print("✅ Ollama infrastructure setup complete!")

📥 1. Installing Ollama on the Colab Linux backend...


>>> Stopping running Ollama instance...
>>> Removing existing Ollama installation...
>>> Downloading Ollama for macOS...
######################################################################## 100.0%
>>> Installing Ollama to /Applications...


🚀 2. Launching Ollama Server in the background...


>>> Starting Ollama...
>>> Install complete. You can now run 'ollama'.


FileNotFoundError: [Errno 2] No such file or directory: 'ollama'

In [7]:
import os
import ast
import re
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextRecall
from openai import OpenAI
from ragas.llms import llm_factory
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_ollama import OllamaEmbeddings

# 1. Inject placeholder key to bypass strict OpenAI client checks
os.environ["OPENAI_API_KEY"] = "ollama-local-bypass"

OLLAMA_MODEL_NAME = "llama3" 
OLLAMA_EMBED_NAME = "nomic-embed-text" 

# 2. Connect to the local Ollama API endpoints
ollama_client = OpenAI(
    api_key="ollama", 
    base_url="http://localhost:11434/v1"
)
evaluator_llm = llm_factory(OLLAMA_MODEL_NAME, provider="openai", client=ollama_client)

ollama_embed = OllamaEmbeddings(model=OLLAMA_EMBED_NAME, base_url="http://localhost:11434")
evaluator_embeddings = LangchainEmbeddingsWrapper(ollama_embed)

# 3. Configure metrics with explicit local model wrappers
metrics = [
    Faithfulness(llm=evaluator_llm),
    AnswerRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings),
    ContextRecall(llm=evaluator_llm)
]

# 4. Fail-safe function to fix malformed stringified lists inside the CSV
CHECKPOINT_FILE = 'progress_checkpoint.csv'

def clean_and_parse_contexts(val):
    if not val or pd.isna(val): 
        return []
    if isinstance(val, list): 
        return val
    val_str = str(val).strip()
    try:
        parsed = ast.literal_eval(val_str)
        if isinstance(parsed, list): 
            return [str(i) for i in parsed]
    except: 
        pass
    matches = re.findall(r"'(.*?)'", val_str)
    if matches: 
        return matches
    return [val_str]

import pandas as pd
from datasets import Dataset

CHECKPOINT_FILE = 'progress_checkpoint.csv'
df_progress = pd.read_csv(CHECKPOINT_FILE)
df_progress['contexts'] = df_progress['contexts'].apply(clean_and_parse_contexts)
df_eval = df_progress[["question", "answer", "contexts", "ground_truth"]]

dataset = Dataset.from_pandas(df_eval)

print("✅ CSV IMPORT !")

# 5. Load and process dataset
if os.path.exists(CHECKPOINT_FILE):
    df_progress = pd.read_csv(CHECKPOINT_FILE)
    df_progress['contexts'] = df_progress['contexts'].apply(clean_and_parse_contexts)
    evaluation_data = df_progress.to_dict(orient='records')
    print(f"🔄 Successfully loaded {len(evaluation_data)} evaluation records from storage.")
else:
    evaluation_data = []
    print("🚨 File Error: Please verify that 'progress_checkpoint.csv' is uploaded to the left sidebar.")

# 6. Execute RAGAS evaluation with asynchronous optimization
if evaluation_data:
    df_eval = pd.DataFrame(evaluation_data)[["question", "answer", "contexts", "ground_truth"]]
    dataset = Dataset.from_pandas(df_eval)

    print("\n📊 Starting RAGAS Evaluation on Colab T4 GPU...")
    try:
        result = evaluate(
            dataset=dataset,
            metrics=metrics,
            llm=evaluator_llm,
            embeddings=evaluator_embeddings,
            is_async=True,  # Enables parallel asynchronous API calls
            batch_size=4    # Safe batch allocation size optimized for T4 VRAM
        )

        print("\n=== ✨ Final Evaluation Summary ===")
        print(result)

        # Export result to CSV
        df_result = result.to_pandas()
        df_result.to_csv('rag_evaluation_report.csv', index=False)
        print("\n✅ Process finished! 'rag_evaluation_report.csv' has been generated.")
        print("💡 Click the 'Refresh' button in the file explorer tab on the left to download your report!")
        
    except Exception as e:
        print(f"🚨 An error occurred during evaluation execution: {e}")

/var/folders/b3/4qrylwt92hz6d_yvrfn7fhxh0000gn/T/ipykernel_56000/462273876.py:7: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextRecall
/var/folders/b3/4qrylwt92hz6d_yvrfn7fhxh0000gn/T/ipykernel_56000/462273876.py:7: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextRecall
/var/folders/b3/4qrylwt92hz6d_yvrfn7fhxh0000gn/T/ipykernel_56000/462273876.py:7: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas

FileNotFoundError: [Errno 2] No such file or directory: 'progress_checkpoint.csv'

In [8]:
import json
import requests
import pandas as pd
import os
import ast
from datasets import Dataset
from ragas import evaluate

# Import Latest RAGAS Metrics Classes 
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextRecall

# 2. Inject Dummy OpenAI Key to Bypass Framework Check
os.environ["OPENAI_API_KEY"] = "ollama-local-bypass"

# 3. Latest Ollama Native Integration & Metrics Setup (CRITICAL FIX)
from openai import OpenAI
from ragas.llms import llm_factory
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_ollama import OllamaEmbeddings  # Requires: pip install langchain-ollama

# Specify the exact model names you have downloaded in Ollama
OLLAMA_MODEL_NAME = "llama3" 
OLLAMA_EMBED_NAME = "nomic-embed-text"

# Setup local Ollama client via OpenAI v1 compatibility endpoint
ollama_client = OpenAI(
    api_key="ollama", 
    base_url="http://localhost:11434/v1"
)
evaluator_llm = llm_factory(OLLAMA_MODEL_NAME, provider="openai", client=ollama_client)

# Wrap the Ollama Embedding model for Ragas compatibility
ollama_embed = OllamaEmbeddings(model=OLLAMA_EMBED_NAME, base_url="http://localhost:11434")
evaluator_embeddings = LangchainEmbeddingsWrapper(ollama_embed)

# 💡 [CRITICAL FIX] Explicitly inject your local LLM and Embeddings into each metric.
# This prevents Ragas from making stealth OpenAI API calls or throwing structure validation errors.
metrics = [
    Faithfulness(llm=evaluator_llm),
    AnswerRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings),
    ContextRecall(llm=evaluator_llm)
]

print(f"🤖 Ollama Model Setup Complete! (LLM: {OLLAMA_MODEL_NAME}, Embed: {OLLAMA_EMBED_NAME})")

# Backend API 
API_URL = "http://localhost:8000/ask"
CHECKPOINT_FILE = 'progress_checkpoint.csv'

# 1. Load Golden Set
if os.path.exists('golden_set.json'):
    with open('golden_set.json', 'r', encoding='utf-8') as f:
        golden_set = json.load(f)
else:
    golden_set = []

# 2. Check for intermediate progress checkpoint (Resume logic)
if os.path.exists(CHECKPOINT_FILE):
    df_progress = pd.read_csv(CHECKPOINT_FILE)
    evaluation_data = df_progress.to_dict(orient='records')
    for item in evaluation_data:
        if isinstance(item['contexts'], str):
            try:
                item['contexts'] = ast.literal_eval(item['contexts'])
            except:
                item['contexts'] = [item['contexts']]
                
    completed_ids = set(df_progress['id'].tolist())
    print(f"🔄 Checkpoint found! Skipping the {len(completed_ids)} already processed items and resuming...")
else:
    evaluation_data = []
    completed_ids = set()

print("🚀 Starting RAG Benchmark Evaluation...")

# 3. Iterate through 50 questions (Will be skipped since all 50 are already collected)
for item in golden_set:
    q_id = item['id']
    query = item['question']
    
    if q_id in completed_ids:
        continue
        
    print(f"Testing {q_id}: {query[:40]}...")
    
    try:
        response = requests.post(API_URL, json={"query": query}, timeout=120)
        if response.status_code == 200:
            data = response.json()
            retrieved_contexts = data.get('sources', [])
            chatbot_answer = data.get('answer', '')
            
            new_row = {
                "id": q_id,
                "question": query,
                "answer": chatbot_answer,
                "contexts": retrieved_contexts,
                "ground_truth": item['ground_truth']
            }
            evaluation_data.append(new_row)
            completed_ids.add(q_id)
            
            df_temp = pd.DataFrame(evaluation_data)
            df_temp.to_csv(CHECKPOINT_FILE, index=False)
        else:
            print(f"❌ Error at {q_id}: Status Code {response.status_code}.")
    except Exception as e:
        print(f"💥 Request failed at {q_id}: {e}.")

# 4. Prepare RAGAS Evaluation
if not evaluation_data:
    print("🚨 No evaluation data collected.")
else:
    df_eval = pd.DataFrame(evaluation_data)[["question", "answer", "contexts", "ground_truth"]]
    dataset = Dataset.from_pandas(df_eval)

    # 5. Run RAGAS Evaluation
    print(f"\n📊 Starting RAGAS Scores calculation for {len(df_eval)} items using Ollama (This might take a few minutes)...")
    try:
        result = evaluate(
            dataset=dataset,
            metrics=metrics,
            llm=evaluator_llm,
            embeddings=evaluator_embeddings
        )

        # 6. Output and save final report
        print("\n=== Final Evaluation Results ===")
        print(result)

        df_result = result.to_pandas()
        df_result.to_csv('rag_evaluation_report.csv', index=False)
        print("✅ Evaluation complete! Report saved to 'rag_evaluation_report.csv'")
        
        # Clean up the checkpoint file once successfully completed
        if os.path.exists(CHECKPOINT_FILE):
            os.remove(CHECKPOINT_FILE)
            
    except Exception as eval_err:
        print(f"🚨 An error occurred during RAGAS evaluation: {eval_err}")
    

/var/folders/b3/4qrylwt92hz6d_yvrfn7fhxh0000gn/T/ipykernel_56000/3372156914.py:10: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextRecall
/var/folders/b3/4qrylwt92hz6d_yvrfn7fhxh0000gn/T/ipykernel_56000/3372156914.py:10: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextRecall
/var/folders/b3/4qrylwt92hz6d_yvrfn7fhxh0000gn/T/ipykernel_56000/3372156914.py:10: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from

🤖 Ollama Model Setup Complete! (LLM: llama3, Embed: nomic-embed-text)
🚀 Starting RAG Benchmark Evaluation...
Testing Q01: What is the minimum number of IEEE SMC S...
Testing Q02: How many Vice-Chairs is a specialised co...
Testing Q03: How long is the standard term of appoint...
Testing Q04: How long is the probationary period for ...
Testing Q05: What percentage of annual TC meetings mu...
Testing Q06: After how many consecutive years of fail...
Testing Q07: For how long must a specialised committe...
Testing Q08: What is the maximum total number of year...
Testing Q09: Is an Associate Editor allowed to serve ...
Testing Q10: What level of Board of Governors approva...
Testing Q11: What is the standard appointment length ...
💥 Request failed at Q11: HTTPConnectionPool(host='localhost', port=8000): Read timed out. (read timeout=120).
Testing Q12: How many signatures are required to peti...
Testing Q13: How many signatures are needed to establ...
Testing Q14: Under what conditions is an

Evaluating:   0%|          | 0/147 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g


=== Final Evaluation Results ===
{'faithfulness': 0.5613, 'answer_relevancy': 0.6439, 'context_recall': 0.7976}
✅ Evaluation complete! Report saved to 'rag_evaluation_report.csv'


In [ ]:
import json
import requests
import pandas as pd
import os
import ast
import warnings
from datasets import Dataset
from ragas import evaluate

# 터미널 Warning 로그 깔끔하게 정리
warnings.filterwarnings("ignore")

# Import Latest RAGAS Metrics Classes 
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextRecall

# 2. Inject Dummy OpenAI Key to Bypass Framework Check
os.environ["OPENAI_API_KEY"] = "ollama-local-bypass"

# 3. Latest Ollama Native Integration & Metrics Setup
from openai import OpenAI
from ragas.llms import llm_factory
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_ollama import OllamaEmbeddings  

# Specify the exact model names you have downloaded in Ollama
OLLAMA_MODEL_NAME = "llama3" 
OLLAMA_EMBED_NAME = "nomic-embed-text"

# Setup local Ollama client via OpenAI v1 compatibility endpoint
ollama_client = OpenAI(
    api_key="ollama", 
    base_url="http://localhost:11434/v1"
)
evaluator_llm = llm_factory(OLLAMA_MODEL_NAME, provider="openai", client=ollama_client)

# Wrap the Ollama Embedding model for Ragas compatibility
ollama_embed = OllamaEmbeddings(model=OLLAMA_EMBED_NAME, base_url="http://localhost:11434")
evaluator_embeddings = LangchainEmbeddingsWrapper(ollama_embed)

# Explicitly inject local LLM and Embeddings into each metric.
metrics = [
    Faithfulness(llm=evaluator_llm),
    AnswerRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings),
    ContextRecall(llm=evaluator_llm)
]

print(f"🤖 Ollama Model Setup Complete! (LLM: {OLLAMA_MODEL_NAME}, Embed: {OLLAMA_EMBED_NAME})")

# Backend API 
API_URL = "http://localhost:8000/ask"
CHECKPOINT_FILE = 'progress_checkpoint.csv'

# 1. Load Golden Set
if os.path.exists('golden_set.json'):
    with open('golden_set.json', 'r', encoding='utf-8') as f:
        golden_set = json.load(f)
        
    golden_set = [item for item in golden_set if int(item['id'].replace('Q', '')) <= 20]
    print(f"🎯 범위 내에 있는 {len(golden_set)}개의 문제만 선별하여 평가를 진행합니다.")
else:
    golden_set = []

# 2. Check for intermediate progress checkpoint (Resume logic)
if os.path.exists(CHECKPOINT_FILE):
    df_progress = pd.read_csv(CHECKPOINT_FILE)
    evaluation_data = df_progress.to_dict(orient='records')
    for item in evaluation_data:
        if isinstance(item['contexts'], str):
            try:
                item['contexts'] = ast.literal_eval(item['contexts'])
            except:
                item['contexts'] = [item['contexts']]
                
    completed_ids = set(df_progress['id'].tolist())
    print(f"🔄 Checkpoint found! Skipping the {len(completed_ids)} already processed items and resuming...")
else:
    evaluation_data = []
    completed_ids = set()

print("🚀 Starting RAG Benchmark Evaluation...")

# 3. Iterate through questions
for item in golden_set:
    q_id = item['id']
    query = item['question']
    
    if q_id in completed_ids:
        continue
        
    print(f"Testing {q_id}: {query[:40]}...")
    
    try:
        response = requests.post(API_URL, json={"query": query}, timeout=120)
        if response.status_code == 200:
            data = response.json()
            
            # 백엔드 데이터 추출 구조 안정화
            retrieved_contexts = data.get('sources', [])
            chatbot_answer = data.get('answer', '')
            
            # [필수 가공] 만약 retrieved_contexts가 쌩 문자열 리스트가 아니라 딕셔너리 리스트 형태라면 텍스트만 추출
            clean_contexts = []
            if isinstance(retrieved_contexts, list):
                for ctx in retrieved_contexts:
                    if isinstance(ctx, dict):
                        # 백엔드 response 구조에 따라 'text', 'content', 'page_content' 중 맞는 키로 매핑하세요
                        clean_contexts.append(ctx.get('text', ctx.get('page_content', str(ctx))))
                    else:
                        clean_contexts.append(str(ctx))
            else:
                clean_contexts = [str(retrieved_contexts)]
            
            new_row = {
                "id": q_id,
                "question": query,
                "answer": chatbot_answer,
                "contexts": clean_contexts,  # 정제된 List[str] 주입
                "ground_truth": item['ground_truth']
            }
            evaluation_data.append(new_row)
            completed_ids.add(q_id)
            
            df_temp = pd.DataFrame(evaluation_data)
            df_temp.to_csv(CHECKPOINT_FILE, index=False)
        else:
            print(f"❌ Error at {q_id}: Status Code {response.status_code}.")
    except Exception as e:
        print(f"💥 Request failed at {q_id}: {e}.")

# ==========================================
# 4. Prepare RAGAS Evaluation (정규화 및 안전벨트 매는 구간 🚨)
# ==========================================
if not evaluation_data:
    print("🚨 No evaluation data collected.")
else:
    df_eval = pd.DataFrame(evaluation_data)
    
    # [컬럼명 일치 보정] 혹시라도 데이터프레임 내 컬럼명이 꼬여있다면 Ragas 표준 규격으로 강제 강제 리네임
    rename_map = {
        'user_input': 'question',
        'query': 'question',
        'response': 'answer',
        'sources': 'contexts'
    }
    df_eval = df_eval.rename(columns={k: v for k, v in rename_map.items() if k in df_eval.columns})
    
    # Ragas가 요구하는 4대 필수 컬럼 명시적 추출
    required_cols = ["question", "answer", "contexts", "ground_truth"]
    df_eval_final = df_eval[required_cols]
    
    # Dataset 객체 생성
    dataset = Dataset.from_pandas(df_eval_final)

    # 5. Run RAGAS Evaluation
    print(f"\n📊 Starting RAGAS Scores calculation for {len(df_eval_final)} items using Ollama...")
    try:
        result = evaluate(
            dataset=dataset,
            metrics=metrics,
            llm=evaluator_llm,
            embeddings=evaluator_embeddings
        )

        # 6. Output and save final report
        print("\n=== Final Evaluation Results ===")
        print(result)

        df_result = result.to_pandas()
        df_result.to_csv('rag_evaluation_report.csv', index=False)
        print("✅ Evaluation complete! Report saved to 'rag_evaluation_report.csv'")
        
        # 성공 완주 시에만 체크포인트 클리어
        if os.path.exists(CHECKPOINT_FILE):
            os.remove(CHECKPOINT_FILE)
            
    except Exception as eval_err:
        print(f"🚨 An error occurred during RAGAS evaluation: {eval_err}")

🤖 Ollama Model Setup Complete! (LLM: llama3, Embed: nomic-embed-text)
🎯 범위 내에 있는 20개의 문제만 선별하여 평가를 진행합니다.
🚀 Starting RAG Benchmark Evaluation...
Testing Q01: What is the minimum number of IEEE SMC S...
Testing Q02: How many Vice-Chairs is a specialised co...
Testing Q03: How long is the standard term of appoint...
Testing Q04: How long is the probationary period for ...
Testing Q05: What percentage of annual TC meetings mu...
Testing Q06: After how many consecutive years of fail...
Testing Q07: For how long must a specialised committe...
Testing Q08: What is the maximum total number of year...
Testing Q09: Is an Associate Editor allowed to serve ...
Testing Q10: What level of Board of Governors approva...
Testing Q11: What is the standard appointment length ...
Testing Q12: How many signatures are required to peti...
Testing Q13: How many signatures are needed to establ...
Testing Q14: Under what conditions is an SMCS Chapter...
Testing Q15: Within how many days of completing trave...


Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g


=== Final Evaluation Results ===
{'faithfulness': 0.6287, 'answer_relevancy': 0.7255, 'context_recall': 0.9333}
✅ Evaluation complete! Report saved to 'rag_evaluation_report.csv'
